# Machine Failure Prediction

Binary classification of machine failures with Logistic Regression, KNN and Decision Trees.

Author: Shirel Azeroual

##התקנת הספריות וטעינת הספריות הדרושות

In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

#חלק 1- טעינת הנתונים והכנת הנתונים

##סעיף 1- טעינת קובץ הנתונים train.csv

In [39]:
df = pd.read_csv("train.csv")

df.head()

,id,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,0,L50096,L,300.6,309.6,1596,36.1,140,0,0,0,0,0,0
1,1,M20343,M,302.6,312.1,1759,29.1,200,0,0,0,0,0,0
2,2,L49454,L,299.3,308.5,1805,26.5,25,0,0,0,0,0,0
3,3,L53355,L,301.0,310.9,1524,44.3,197,0,0,0,0,0,0
4,4,M24050,M,298.0,309.0,1641,35.4,34,0,0,0,0,0,0


##סעיף 2 - מחיקת העמודות id ו- Product ID

In [40]:
df = df.drop(columns=['id', 'Product ID'])

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,L,300.6,309.6,1596,36.1,140,0,0,0,0,0,0
1,M,302.6,312.1,1759,29.1,200,0,0,0,0,0,0
2,L,299.3,308.5,1805,26.5,25,0,0,0,0,0,0
3,L,301.0,310.9,1524,44.3,197,0,0,0,0,0,0
4,M,298.0,309.0,1641,35.4,34,0,0,0,0,0,0


## סעיף 3 - הצגת כמות הרשומות מכל קלאס

In [41]:
print(df['Machine failure'].value_counts())

Machine failure
0    134281
1      2148
Name: count, dtype: int64


ניתן לראות כי הדאטה אינו מאוזן וקיימות הרבה יותר רשומות של מכונות תקינות לעומת מכונות עם כשל.

## סעיף 4 - המרת המשתנה הקטגוריאלי Type לערכים מספריים

In [42]:
df['Type'] = df['Type'].map({
    'H': 0,
    'M': 1,
    'L': 2
})

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,2,300.6,309.6,1596,36.1,140,0,0,0,0,0,0
1,1,302.6,312.1,1759,29.1,200,0,0,0,0,0,0
2,2,299.3,308.5,1805,26.5,25,0,0,0,0,0,0
3,2,301.0,310.9,1524,44.3,197,0,0,0,0,0,0
4,1,298.0,309.0,1641,35.4,34,0,0,0,0,0,0


## סעיף 5 - נרמול הנתונים באמצעות MinMaxScaler

In [43]:
X = df.drop('Machine failure', axis=1)

y = df['Machine failure']

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

## סעיף 6 - חלוקת הנתונים ל-Train ו-Test

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(109143, 11)
(27286, 11)


# חלק 2 - אימון מודל רגרסיה לוגיסטית

## סעיף 7.1 - אימון מודל Logistic Regression והצגת מקדמי הרגרסיה

In [45]:
# יצירת מודל רגרסיה לוגיסטית
log_model = LogisticRegression(max_iter=1000)

# אימון המודל
log_model.fit(X_train, y_train)

# הצגת מקדמי הרגרסיה
print("Regression Coefficients:")

for feature, coef in zip(X.columns, log_model.coef_[0]):
    print(feature, ":", coef)

Regression Coefficients:
Type : -0.05428717115454673
Air temperature [K] : 1.7691022141355581
Process temperature [K] : -1.3291829943071927
Rotational speed [rpm] : -0.3611402217011631
Torque [Nm] : 3.9264071028283682
Tool wear [min] : 0.8230959226400137
TWF : 8.610553763035378
HDF : 8.73131456180624
PWF : 9.946183021076646
OSF : 8.405335578100981
RNF : 0.006736540733924448


## סעיף 7.2 - חישוב מדדי Accuracy, Precision ו-Recall

In [46]:
# חיזוי על קבוצת האימון
y_train_pred = log_model.predict(X_train)

# חיזוי על קבוצת הבדיקה
y_test_pred = log_model.predict(X_test)

# מדדי Train
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)

# מדדי Test
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)

# הצגת התוצאות
print("Train Metrics:")
print("Accuracy:", train_accuracy)
print("Precision:", train_precision)
print("Recall:", train_recall)

print()

print("Test Metrics:")
print("Accuracy:", test_accuracy)
print("Precision:", test_precision)
print("Recall:", test_recall)

Train Metrics:
Accuracy: 0.9961884866642845
Precision: 0.9923896499238964
Recall: 0.7625730994152047

Test Metrics:
Accuracy: 0.9961518727552591
Precision: 0.9911504424778761
Recall: 0.7671232876712328


## סעיף 7.3 - הצגת Confusion Matrix

In [47]:
# Confusion Matrix עבור Train
train_cm = confusion_matrix(y_train, y_train_pred)

print("Train Confusion Matrix:")
print(train_cm)

print()

# Confusion Matrix עבור Test
test_cm = confusion_matrix(y_test, y_test_pred)

print("Test Confusion Matrix:")
print(test_cm)

Train Confusion Matrix:
[[107423     10]
 [   406   1304]]

Test Confusion Matrix:
[[26845     3]
 [  102   336]]


# חלק 3 - Logistic Regression לאחר הסרת משתנים בינאריים

## סעיף 8 - הסרת המשתנים הבינאריים ואימון המודל מחדש
בסעיף זה נוריד את המשתנים הבינאריים TWF, HDF, PWF, OSF, RNF ונאמן מחדש מודל Logistic Regression.

In [48]:
# יצירת עותק של הנתונים בלי המשתנים הבינאריים
df_without_binary = df.drop(columns=['TWF', 'HDF', 'PWF', 'OSF', 'RNF'])

# הגדרת X ו-y מחדש
X_without_binary = df_without_binary.drop('Machine failure', axis=1)
y_without_binary = df_without_binary['Machine failure']

# נרמול מחדש
scaler_without_binary = MinMaxScaler()
X_without_binary_scaled = scaler_without_binary.fit_transform(X_without_binary)

# חלוקה מחדש ל-train ו-test
X_train_wb, X_test_wb, y_train_wb, y_test_wb = train_test_split(
    X_without_binary_scaled,
    y_without_binary,
    test_size=0.2,
    random_state=42
)

# יצירת מודל Logistic Regression חדש
log_model_wb = LogisticRegression(max_iter=1000)

# אימון המודל
log_model_wb.fit(X_train_wb, y_train_wb)

# חיזוי
y_train_pred_wb = log_model_wb.predict(X_train_wb)
y_test_pred_wb = log_model_wb.predict(X_test_wb)

# מדדי Train
train_accuracy_wb = accuracy_score(y_train_wb, y_train_pred_wb)
train_precision_wb = precision_score(y_train_wb, y_train_pred_wb)
train_recall_wb = recall_score(y_train_wb, y_train_pred_wb)

# מדדי Test
test_accuracy_wb = accuracy_score(y_test_wb, y_test_pred_wb)
test_precision_wb = precision_score(y_test_wb, y_test_pred_wb)
test_recall_wb = recall_score(y_test_wb, y_test_pred_wb)

# הצגת מקדמי הרגרסיה
print("Regression Coefficients after removing binary features:")

for feature, coef in zip(X_without_binary.columns, log_model_wb.coef_[0]):
    print(feature, ":", coef)

print("\nTrain Metrics:")
print("Accuracy:", train_accuracy_wb)
print("Precision:", train_precision_wb)
print("Recall:", train_recall_wb)

print("\nTest Metrics:")
print("Accuracy:", test_accuracy_wb)
print("Precision:", test_precision_wb)
print("Recall:", test_recall_wb)

print("\nTrain Confusion Matrix:")
print(confusion_matrix(y_train_wb, y_train_pred_wb))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test_wb, y_test_pred_wb))

Regression Coefficients after removing binary features:
Type : 0.3670466860767656
Air temperature [K] : 5.921672610520267
Process temperature [K] : -4.545340400842117
Rotational speed [rpm] : 4.680724339097804
Torque [Nm] : 11.450597834988402
Tool wear [min] : 1.8366345252933396

Train Metrics:
Accuracy: 0.9846531614487416
Precision: 0.7692307692307693
Recall: 0.029239766081871343

Test Metrics:
Accuracy: 0.9842043538811112
Precision: 0.7692307692307693
Recall: 0.0228310502283105

Train Confusion Matrix:
[[107418     15]
 [  1660     50]]

Test Confusion Matrix:
[[26845     3]
 [  428    10]]


לאחר הסרת המשתנים הבינאריים נצפה שינוי קטן במדדי הדיוק של המודל.

# חלק 4 - בדיקת מודלים נוספים

## סעיף 9.1 - בדיקת מודל KNN עם ערכי k שונים
בסעיף זה נבדוק את מודל KNN עם 4 קונפיגורציות שונות של מספר שכנים.

In [49]:
# רשימה לשמירת התוצאות
results = []

# ערכי k שונים לבדיקה
knn_neighbors = [3, 5, 7, 9]

for k in knn_neighbors:
    # יצירת מודל KNN
    knn_model = KNeighborsClassifier(n_neighbors=k)

    # אימון המודל
    knn_model.fit(X_train_wb, y_train_wb)

    # חיזוי
    y_train_pred_knn = knn_model.predict(X_train_wb)
    y_test_pred_knn = knn_model.predict(X_test_wb)

    # חישוב מדדים
    results.append({
        'Model': 'KNN',
        'Configuration': f'n_neighbors={k}',
        'Train Accuracy': accuracy_score(y_train_wb, y_train_pred_knn),
        'Train Precision': precision_score(y_train_wb, y_train_pred_knn),
        'Train Recall': recall_score(y_train_wb, y_train_pred_knn),
        'Test Accuracy': accuracy_score(y_test_wb, y_test_pred_knn),
        'Test Precision': precision_score(y_test_wb, y_test_pred_knn),
        'Test Recall': recall_score(y_test_wb, y_test_pred_knn),
        'Test Confusion Matrix': confusion_matrix(y_test_wb, y_test_pred_knn)
    })

# הצגת התוצאות של KNN
pd.DataFrame(results)

,Model,Configuration,Train Accuracy,Train Precision,Train Recall,Test Accuracy,Test Precision,Test Recall,Test Confusion Matrix
0,KNN,n_neighbors=3,0.989097,0.786976,0.416959,0.983948,0.500000,0.216895,"[[26753, 95], [343, 95]]"
1,KNN,n_neighbors=5,0.988071,0.787324,0.326901,0.984717,0.572414,0.189498,"[[26786, 62], [355, 83]]"
2,KNN,n_neighbors=7,0.986916,0.739796,0.254386,0.985157,0.624060,0.189498,"[[26798, 50], [355, 83]]"
3,KNN,n_neighbors=9,0.986367,0.735169,0.202924,0.985377,0.678899,0.168950,"[[26813, 35], [364, 74]]"


## סעיף 9.2 - בדיקת מודל Decision Tree עם עומקים שונים
בסעיף זה נבדוק את מודל Decision Tree עם 4 קונפיגורציות שונות של עומק העץ.

In [50]:
# עומקי עץ שונים לבדיקה
tree_depths = [3, 5, 7, 10]

for depth in tree_depths:
    # יצירת מודל Decision Tree
    tree_model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    # אימון המודל
    tree_model.fit(X_train_wb, y_train_wb)

    # חיזוי
    y_train_pred_tree = tree_model.predict(X_train_wb)
    y_test_pred_tree = tree_model.predict(X_test_wb)

    # חישוב מדדים
    results.append({
        'Model': 'Decision Tree',
        'Configuration': f'max_depth={depth}',
        'Train Accuracy': accuracy_score(y_train_wb, y_train_pred_tree),
        'Train Precision': precision_score(y_train_wb, y_train_pred_tree),
        'Train Recall': recall_score(y_train_wb, y_train_pred_tree),
        'Test Accuracy': accuracy_score(y_test_wb, y_test_pred_tree),
        'Test Precision': precision_score(y_test_wb, y_test_pred_tree),
        'Test Recall': recall_score(y_test_wb, y_test_pred_tree),
        'Test Confusion Matrix': confusion_matrix(y_test_wb, y_test_pred_tree)
    })

# הצגת כל התוצאות עד עכשיו
results_df = pd.DataFrame(results)
results_df

,Model,Configuration,Train Accuracy,Train Precision,Train Recall,Test Accuracy,Test Precision,Test Recall,Test Confusion Matrix
0,KNN,n_neighbors=3,0.989097,0.786976,0.416959,0.983948,0.500000,0.216895,"[[26753, 95], [343, 95]]"
1,KNN,n_neighbors=5,0.988071,0.787324,0.326901,0.984717,0.572414,0.189498,"[[26786, 62], [355, 83]]"
2,KNN,n_neighbors=7,0.986916,0.739796,0.254386,0.985157,0.624060,0.189498,"[[26798, 50], [355, 83]]"
3,KNN,n_neighbors=9,0.986367,0.735169,0.202924,0.985377,0.678899,0.168950,"[[26813, 35], [364, 74]]"
4,Decision Tree,max_depth=3,0.985661,0.769517,0.121053,0.985047,0.714286,0.114155,"[[26828, 20], [388, 50]]"
5,Decision Tree,max_depth=5,0.986486,0.670537,0.270175,0.985524,0.614973,0.262557,"[[26776, 72], [323, 115]]"
6,Decision Tree,max_depth=7,0.987759,0.703704,0.377778,0.985377,0.582278,0.315068,"[[26749, 99], [300, 138]]"
7,Decision Tree,max_depth=10,0.989986,0.827843,0.455556,0.985011,0.556863,0.324201,"[[26735, 113], [296, 142]]"


# חלק 5 - טבלת השוואת המודלים

## סעיף 10 - השוואת ביצועי המודלים
בטבלה הבאה מוצגים מדדי הדיוק של כל המודלים והקונפיגורציות שנבדקו.

In [51]:
# הצגת רק המדדים המרכזיים
comparison_table_clean = comparison_table[[
    'Model',
    'Configuration',
    'Train Accuracy',
    'Test Accuracy',
    'Train Precision',
    'Test Precision',
    'Train Recall',
    'Test Recall'
]]

comparison_table_clean

,Model,Configuration,Train Accuracy,Test Accuracy,Train Precision,Test Precision,Train Recall,Test Recall
0,KNN,n_neighbors=3,0.989097,0.983948,0.786976,0.500000,0.416959,0.216895
1,KNN,n_neighbors=5,0.988071,0.984717,0.787324,0.572414,0.326901,0.189498
2,KNN,n_neighbors=7,0.986916,0.985157,0.739796,0.624060,0.254386,0.189498
3,KNN,n_neighbors=9,0.986367,0.985377,0.735169,0.678899,0.202924,0.168950
4,Decision Tree,max_depth=3,0.985661,0.985047,0.769517,0.714286,0.121053,0.114155
5,Decision Tree,max_depth=5,0.986486,0.985524,0.670537,0.614973,0.270175,0.262557
6,Decision Tree,max_depth=7,0.987759,0.985377,0.703704,0.582278,0.377778,0.315068
7,Decision Tree,max_depth=10,0.989986,0.985011,0.827843,0.556863,0.455556,0.324201


# חלק 6 - בחירת המודלים המובילים

## סעיף 11 - בחירת המודלים הטובים ביותר
נבחרו המודלים שהציגו את תוצאות ה-Test הטובות ביותר לפי Accuracy, Precision ו-Recall.

In [52]:
# מיון לפי Test Accuracy
top_models = comparison_table.sort_values(
    by='Test Accuracy',
    ascending=False
)

# הצגת 3 המודלים הטובים ביותר
top_models.head(3)

,Model,Configuration,Train Accuracy,Train Precision,Train Recall,Test Accuracy,Test Precision,Test Recall,Test Confusion Matrix
5,Decision Tree,max_depth=5,0.986486,0.670537,0.270175,0.985524,0.614973,0.262557,"[[26776, 72], [323, 115]]"
3,KNN,n_neighbors=9,0.986367,0.735169,0.202924,0.985377,0.678899,0.168950,"[[26813, 35], [364, 74]]"
6,Decision Tree,max_depth=7,0.987759,0.703704,0.377778,0.985377,0.582278,0.315068,"[[26749, 99], [300, 138]]"


המודלים שנבחרו הם המודלים שהציגו את ביצועי ה-Test הטובים ביותר.
הבחירה בוצעה לפי מדדי Accuracy, Precision ו-Recall.

## Confusion Matrix של המודלים המובילים

In [53]:
# הצגת confusion matrix של 3 המודלים הטובים ביותר
for i in range(3):
    print("Model:")
    print(top_models.iloc[i]['Model'])

    print("Configuration:")
    print(top_models.iloc[i]['Configuration'])

    print("Confusion Matrix:")
    print(top_models.iloc[i]['Test Confusion Matrix'])

    print("\n----------------------\n")

Model:
Decision Tree
Configuration:
max_depth=5
Confusion Matrix:
[[26776    72]
 [  323   115]]

----------------------

Model:
KNN
Configuration:
n_neighbors=9
Confusion Matrix:
[[26813    35]
 [  364    74]]

----------------------

Model:
Decision Tree
Configuration:
max_depth=7
Confusion Matrix:
[[26749    99]
 [  300   138]]

----------------------



המודלים שנבחרו הם המודלים שהציגו את תוצאות ה-Test הטובות ביותר לפי Accuracy, Precision ו-Recall.

# חלק 7 - השפעת הסרת המשתנה החשוב ביותר

## סעיף 12 - מציאת המשתנה החשוב ביותר במודל הרגרסיה
בסעיף זה נמצא את המשתנה עם החשיבות הגבוהה ביותר לפי מקדמי מודל Logistic Regression.

In [54]:
# מציאת המשתנה עם המקדם הגבוה ביותר
coefficients = pd.DataFrame({
    'Feature': X_without_binary.columns,
    'Coefficient': abs(log_model_wb.coef_[0])
})

# מיון לפי חשיבות
coefficients = coefficients.sort_values(
    by='Coefficient',
    ascending=False
)

# הצגת המשתנים
coefficients

,Feature,Coefficient
4,Torque [Nm],11.450598
1,Air temperature [K],5.921673
3,Rotational speed [rpm],4.680724
2,Process temperature [K],4.545340
5,Tool wear [min],1.836635
0,Type,0.367047


In [55]:
# המשתנה החשוב ביותר
most_important_feature = coefficients.iloc[0]['Feature']

print("Most Important Feature:")
print(most_important_feature)

Most Important Feature:
Torque [Nm]


## הסרת המשתנה החשוב ביותר ואימון מחדש של המודלים

In [56]:
# הסרת המשתנה החשוב ביותר
df_final = df_without_binary.drop(columns=[most_important_feature])

# הגדרת X ו-y מחדש
X_final = df_final.drop('Machine failure', axis=1)
y_final = df_final['Machine failure']

# נרמול
scaler_final = MinMaxScaler()
X_final_scaled = scaler_final.fit_transform(X_final)

# חלוקה מחדש
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final_scaled,
    y_final,
    test_size=0.2,
    random_state=42
)

## אימון מחדש של Logistic Regression

In [57]:
# יצירת מודל Logistic Regression
final_log_model = LogisticRegression(max_iter=1000)

# אימון
final_log_model.fit(X_train_final, y_train_final)

# חיזוי
y_test_pred_final_log = final_log_model.predict(X_test_final)

# מדדים
final_log_accuracy = accuracy_score(y_test_final, y_test_pred_final_log)
final_log_precision = precision_score(y_test_final, y_test_pred_final_log)
final_log_recall = recall_score(y_test_final, y_test_pred_final_log)

print("Logistic Regression After Removing Feature")

print("Accuracy:", final_log_accuracy)
print("Precision:", final_log_precision)
print("Recall:", final_log_recall)

print("Confusion Matrix:")
print(confusion_matrix(y_test_final, y_test_pred_final_log))

Logistic Regression After Removing Feature
Accuracy: 0.9839478120647951
Precision: 0.0
Recall: 0.0
Confusion Matrix:
[[26848     0]
 [  438     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## אימון מחדש של Decision Tree

In [58]:
# Decision Tree
final_tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

# אימון
final_tree_model.fit(X_train_final, y_train_final)

# חיזוי
y_test_pred_final_tree = final_tree_model.predict(X_test_final)

# מדדים
final_tree_accuracy = accuracy_score(y_test_final, y_test_pred_final_tree)
final_tree_precision = precision_score(y_test_final, y_test_pred_final_tree)
final_tree_recall = recall_score(y_test_final, y_test_pred_final_tree)

print("Decision Tree After Removing Feature")

print("Accuracy:", final_tree_accuracy)
print("Precision:", final_tree_precision)
print("Recall:", final_tree_recall)

print("Confusion Matrix:")
print(confusion_matrix(y_test_final, y_test_pred_final_tree))

Decision Tree After Removing Feature
Accuracy: 0.9845341933592319
Precision: 0.5421052631578948
Recall: 0.23515981735159816
Confusion Matrix:
[[26761    87]
 [  335   103]]


## השוואת ביצועים לפני ואחרי הסרת המשתנה

In [59]:
# טבלת השוואה
comparison_after_removal = pd.DataFrame({

    'Model': [
        'Logistic Regression Before',
        'Logistic Regression After',
        'Decision Tree Before',
        'Decision Tree After'
    ],

    'Accuracy': [
        test_accuracy_wb,
        final_log_accuracy,
        top_models.iloc[0]['Test Accuracy'],
        final_tree_accuracy
    ],

    'Precision': [
        test_precision_wb,
        final_log_precision,
        top_models.iloc[0]['Test Precision'],
        final_tree_precision
    ],

    'Recall': [
        test_recall_wb,
        final_log_recall,
        top_models.iloc[0]['Test Recall'],
        final_tree_recall
    ]
})

comparison_after_removal

,Model,Accuracy,Precision,Recall
0,Logistic Regression Before,0.984204,0.769231,0.022831
1,Logistic Regression After,0.983948,0.000000,0.000000
2,Decision Tree Before,0.985524,0.614973,0.262557
3,Decision Tree After,0.984534,0.542105,0.235160


הסרת המשתנה החשוב ביותר השפיעה על ביצועי המודלים וניתן לראות את ההבדלים במדדי הדיוק בטבלה.